In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib
matplotlib.use('Agg')
import os
os.chdir('C:/Users/Lenovo/churn-predictor')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import pickle
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.calibration import (
    CalibratedClassifierCV,
    CalibrationDisplay,
    calibration_curve,
)
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    log_loss,
)

from src.features import run_pipeline, build_preprocessor

# Load data
X_train, X_val, X_test, y_train, y_val, y_test = run_pipeline(
    'data/raw/telco_churn.csv'
)

# Load best params
with open('models/best_params_lgbm.json') as f:
    config = json.load(f)

# Train base pipeline
pipeline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', LGBMClassifier(
        **config['best_params'],
        is_unbalance=True,
        metric='average_precision',
        random_state=42,
        verbose=-1,
    ))
])
pipeline.fit(X_train, y_train)

y_prob_val = pipeline.predict_proba(X_val)[:, 1]

print("Pipeline trained.")
print(f"Val ROC-AUC : {roc_auc_score(y_val, y_prob_val):.4f}")
print(f"Val PR-AUC  : {average_precision_score(y_val, y_prob_val):.4f}")

Train : (4929, 21) | churn rate: 0.265
Val   : (1057, 21)   | churn rate: 0.266
Test  : (1057, 21)  | churn rate: 0.265


C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Pipeline trained.
Val ROC-AUC : 0.8333
Val PR-AUC  : 0.6521


In [2]:
def plot_calibration(y_true, y_prob, label, ax, n_bins=10):
    """
    Plot a calibration curve on a given axes.
    Also returns the mean calibration error.
    """
    prob_true, prob_pred = calibration_curve(
        y_true, y_prob,
        n_bins=n_bins,
        strategy='uniform',
    )

    ax.plot(prob_pred, prob_true,
            marker='o', linewidth=2, label=label)
    ax.plot([0, 1], [0, 1],
            'k--', alpha=0.5, label='Perfect calibration')
    ax.set_xlabel('Mean predicted probability', fontsize=10)
    ax.set_ylabel('Fraction of positives', fontsize=10)
    ax.set_title('Calibration Curve', fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    # Mean calibration error
    mce = np.mean(np.abs(prob_true - prob_pred))
    return mce


fig, ax = plt.subplots(figsize=(7, 6))
mce = plot_calibration(y_val, y_prob_val, 'LightGBM (uncalibrated)', ax)

plt.tight_layout()
plt.savefig('reports/figures/calibration_base.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean calibration error (base): {mce:.4f}")
print(f"Brier score (base):            {brier_score_loss(y_val, y_prob_val):.4f}")

Mean calibration error (base): 0.1553
Brier score (base):            0.1609


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22860\3387513812.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Calibration curve — base model

LightGBM with is_unbalance=True tends to be under-confident —
it assigns lower probabilities than the true churn rate would justify.

This is a known property of tree ensembles: because they use voting
across many trees, extreme probabilities (0.05, 0.95) are rare.
The distribution clusters toward 0.3–0.7.

Mean calibration error measures the average gap between predicted
probability and actual fraction of churners — lower is better.
Brier score is a proper scoring rule that combines calibration and
sharpness — also lower is better.

In [3]:
from sklearn.calibration import CalibratedClassifierCV

# CalibratedClassifierCV wraps the entire pipeline
# method='sigmoid' is Platt scaling
# cv='prefit' means the pipeline is already fitted — don't refit
calibrated_pipeline = CalibratedClassifierCV(
    pipeline,
    method='sigmoid',
    cv='prefit',
)

# Fit the calibrator on val set
# Note: in production you'd use a separate calibration set
# Using val here is a pragmatic choice for a 7K row dataset
calibrated_pipeline.fit(X_val, y_val)

y_prob_val_calibrated = calibrated_pipeline.predict_proba(X_val)[:, 1]

print("Platt scaling calibration fitted.")
print(f"Brier score (calibrated): {brier_score_loss(y_val, y_prob_val_calibrated):.4f}")

Platt scaling calibration fitted.
Brier score (calibrated): 0.1394


C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [4]:
calibrated_pipeline_iso = CalibratedClassifierCV(
    pipeline,
    method='isotonic',
    cv='prefit',
)
calibrated_pipeline_iso.fit(X_val, y_val)

y_prob_val_iso = calibrated_pipeline_iso.predict_proba(X_val)[:, 1]

print("Isotonic calibration fitted.")
print(f"Brier score (isotonic): {brier_score_loss(y_val, y_prob_val_iso):.4f}")

Isotonic calibration fitted.
Brier score (isotonic): 0.1337


C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

variants = [
    (y_prob_val,             'LightGBM (uncalibrated)', 'steelblue'),
    (y_prob_val_calibrated,  'Platt scaling',           'tomato'),
    (y_prob_val_iso,         'Isotonic regression',     'seagreen'),
]

# Left: calibration curves
for y_prob, label, color in variants:
    prob_true, prob_pred = calibration_curve(
        y_val, y_prob, n_bins=10, strategy='uniform'
    )
    axes[0].plot(prob_pred, prob_true,
                 marker='o', color=color, linewidth=2, label=label)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
axes[0].set_xlabel('Mean predicted probability', fontsize=10)
axes[0].set_ylabel('Fraction of positives', fontsize=10)
axes[0].set_title('Calibration Curves — Comparison', fontweight='bold')
axes[0].legend(fontsize=9)

# Right: probability distribution histograms
for y_prob, label, color in variants:
    axes[1].hist(y_prob, bins=30, alpha=0.4, color=color,
                 label=label, density=True, edgecolor='white')

axes[1].axvline(x=y_val.mean(), color='black', linestyle='--',
                alpha=0.7, label=f'True churn rate ({y_val.mean():.2f})')
axes[1].set_xlabel('Predicted probability', fontsize=10)
axes[1].set_ylabel('Density', fontsize=10)
axes[1].set_title('Probability Distribution', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Calibration Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/calibration_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22860\2285602597.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
rows = []
for y_prob, label, color in variants:      # ← unpack all three
    threshold = 0.35
    y_pred = (y_prob >= threshold).astype(int)

    prob_true, prob_pred = calibration_curve(
        y_val, y_prob, n_bins=10, strategy='uniform'
    )
    mce = np.mean(np.abs(prob_true - prob_pred))

    rows.append({
        'Model'        : label,
        'Brier score'  : round(brier_score_loss(y_val, y_prob), 4),
        'Log loss'     : round(log_loss(y_val, y_prob), 4),
        'MCE'          : round(mce, 4),
        'ROC-AUC'      : round(roc_auc_score(y_val, y_prob), 4),
        'PR-AUC'       : round(average_precision_score(y_val, y_prob), 4),
        'F1'           : round(f1_score(y_val, y_pred), 4),
    })

calib_df = pd.DataFrame(rows).set_index('Model')
print("\nCalibration metrics comparison:")
print("="*75)
print(calib_df.to_string())
print("="*75)
print("\nLower Brier score and Log loss = better calibration")
print("ROC-AUC and PR-AUC should be identical or very close across all three")


Calibration metrics comparison:
                         Brier score  Log loss     MCE  ROC-AUC  PR-AUC      F1
Model                                                                          
LightGBM (uncalibrated)       0.1609    0.4837  0.1553   0.8333  0.6521  0.5943
Platt scaling                 0.1394    0.4345  0.0565   0.8333  0.6521  0.6219
Isotonic regression           0.1337    0.4134  0.0000   0.8417  0.6456  0.6352

Lower Brier score and Log loss = better calibration
ROC-AUC and PR-AUC should be identical or very close across all three


## Calibration metrics interpretation

Brier score: mean squared error between predicted probability and
actual outcome (0 or 1). Range 0-1, lower is better.
A perfectly calibrated model at 26.5% base rate has Brier = 0.265 × 0.735 ≈ 0.19.

Log loss: measures the quality of probability estimates. Heavily penalises
confident wrong predictions. Lower is better.

MCE (Mean Calibration Error): average absolute gap between predicted
probability bins and actual fraction of positives. Lower is better.

Key observation: calibration methods should improve Brier score and
MCE without significantly changing ROC-AUC or PR-AUC.
ROC-AUC is ranking-based and unaffected by probability scale.
PR-AUC can shift slightly because it measures precision at each threshold.

Decision: use Platt scaling if Brier score improves. The improvement
in probability reliability is worth the small additional complexity.

In [8]:
# Compare Brier scores
base_brier     = brier_score_loss(y_val, y_prob_val)
platt_brier    = brier_score_loss(y_val, y_prob_val_calibrated)
isotonic_brier = brier_score_loss(y_val, y_prob_val_iso)

print(f"Brier score comparison:")
print(f"  Base model        : {base_brier:.4f}")
print(f"  Platt scaling     : {platt_brier:.4f}")
print(f"  Isotonic          : {isotonic_brier:.4f}")

# Choose the best calibrated model
if min(platt_brier, isotonic_brier) < base_brier - 0.005:
    if platt_brier <= isotonic_brier:
        final_pipeline = calibrated_pipeline
        calib_method   = 'platt'
        print("\nDecision: use Platt scaling — meaningful Brier improvement")
    else:
        final_pipeline = calibrated_pipeline_iso
        calib_method   = 'isotonic'
        print("\nDecision: use isotonic regression — meaningful Brier improvement")
else:
    final_pipeline = pipeline
    calib_method   = 'none'
    print("\nDecision: keep base model — calibration improvement is negligible")

print(f"Final model: LightGBM + calibration={calib_method}")

# Save the decision to threshold config
with open('models/threshold_config.json') as f:
    thresh_config = json.load(f)

thresh_config['calibration_method'] = calib_method
thresh_config['base_brier_score']   = round(base_brier, 4)
thresh_config['final_brier_score']  = round(
    brier_score_loss(y_val,
                     final_pipeline.predict_proba(X_val)[:, 1]), 4
)

with open('models/threshold_config.json', 'w') as f:
    json.dump(thresh_config, f, indent=2)

print(f"\nUpdated models/threshold_config.json")
print(json.dumps(thresh_config, indent=2))

Brier score comparison:
  Base model        : 0.1609
  Platt scaling     : 0.1394
  Isotonic          : 0.1337

Decision: use isotonic regression — meaningful Brier improvement
Final model: LightGBM + calibration=isotonic

Updated models/threshold_config.json
{
  "optimal_threshold": 0.621,
  "threshold_metric": "f1",
  "val_f1_at_threshold": 0.6342,
  "val_precision": 0.6,
  "val_recall": 0.6726,
  "note": "Computed on validation set. Adjust for business needs.",
  "calibration_method": "isotonic",
  "base_brier_score": 0.1609,
  "final_brier_score": 0.1337
}


C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [9]:
print("\n" + "="*55)
print("FINAL EVALUATION ON TEST SET")
print("This is the first time the test set is used.")
print("="*55)

y_prob_test = final_pipeline.predict_proba(X_test)[:, 1]

threshold = thresh_config['optimal_threshold']
y_pred_test = (y_prob_test >= threshold).astype(int)

test_metrics = {
    'roc_auc'  : round(roc_auc_score(y_test, y_prob_test),           4),
    'pr_auc'   : round(average_precision_score(y_test, y_prob_test), 4),
    'f1'       : round(f1_score(y_test, y_pred_test),                4),
    'precision': round(precision_score(y_test, y_pred_test),         4),
    'recall'   : round(recall_score(y_test, y_pred_test),            4),
    'brier'    : round(brier_score_loss(y_test, y_prob_test),        4),
}

print(f"\nTest set metrics (threshold={threshold}):")
for k, v in test_metrics.items():
    print(f"  {k:<12} {v:.4f}")

# Compare val vs test to check for overfitting
val_roc  = roc_auc_score(y_val,  final_pipeline.predict_proba(X_val)[:,1])
test_roc = test_metrics['roc_auc']

print(f"\nVal ROC-AUC  : {val_roc:.4f}")
print(f"Test ROC-AUC : {test_roc:.4f}")
print(f"Gap          : {abs(val_roc - test_roc):.4f}")

if abs(val_roc - test_roc) < 0.02:
    print("Gap < 0.02 — no significant overfitting detected.")
else:
    print("Gap >= 0.02 — some overfitting present. Note in model card.")

# Save test metrics
with open('models/test_metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)
print("\nTest metrics saved to models/test_metrics.json")


FINAL EVALUATION ON TEST SET
This is the first time the test set is used.

Test set metrics (threshold=0.621):
  roc_auc      0.8429
  pr_auc       0.6279
  f1           0.4726
  precision    0.7122
  recall       0.3536
  brier        0.1380

Val ROC-AUC  : 0.8417
Test ROC-AUC : 0.8429
Gap          : 0.0012
Gap < 0.02 — no significant overfitting detected.

Test metrics saved to models/test_metrics.json


C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Test set evaluation

Test metrics are the honest, unbiased estimate of production performance.
All prior decisions (feature engineering, model selection, threshold tuning,
calibration) were made on train/val only — the test set has never been seen.

If val-test gap > 0.02 ROC-AUC, note it as a limitation in the model card.
Small gaps (< 0.01) indicate the model generalises well.

These are the numbers that go in the README and model card.

In [10]:
model_card_content = f"""# Model Card — Telco Customer Churn Predictor

## Model Details

| Field | Value |
|---|---|
| Model type | LightGBM gradient boosting classifier |
| Version | 1.0 |
| Training date | May 2026 |
| Author | Your Name |
| Framework | LightGBM {__import__('lightgbm').__version__}, scikit-learn |
| Calibration | {calib_method.title()} scaling |

## Intended Use

**Primary use case:** Identify telecom customers at high risk of churning
so the retention team can proactively offer incentives before cancellation.

**Intended users:** Retention campaign managers, data analysts, CRM teams.

**Out-of-scope uses:**
- Pricing decisions (model was not trained to optimise revenue)
- Automated cancellation or service changes without human review
- Customer segments outside the telecom vertical

## Training Data

| Field | Value |
|---|---|
| Dataset | IBM Telco Customer Churn |
| Source | Kaggle (public dataset) |
| Size | 7,043 customers |
| Positive class (Churn) | 26.5% |
| Features | 19 input features + 2 derived |
| Train split | 4,930 customers (70%) |
| Val split | 1,057 customers (15%) |
| Test split | 1,056 customers (15%) |

## Performance Metrics

### Validation set (used for model selection and tuning)

| Metric | Value |
|---|---|
| ROC-AUC | {round(roc_auc_score(y_val, final_pipeline.predict_proba(X_val)[:,1]),4)} |
| PR-AUC | {round(average_precision_score(y_val, final_pipeline.predict_proba(X_val)[:,1]),4)} |
| F1 | {round(f1_score(y_val, (final_pipeline.predict_proba(X_val)[:,1] >= threshold).astype(int)),4)} |
| Precision | {round(precision_score(y_val, (final_pipeline.predict_proba(X_val)[:,1] >= threshold).astype(int)),4)} |
| Recall | {round(recall_score(y_val, (final_pipeline.predict_proba(X_val)[:,1] >= threshold).astype(int)),4)} |
| Brier score | {thresh_config['final_brier_score']} |

### Test set (honest unbiased estimate — touched once)

| Metric | Value |
|---|---|
| ROC-AUC | {test_metrics['roc_auc']} |
| PR-AUC | {test_metrics['pr_auc']} |
| F1 | {test_metrics['f1']} |
| Precision | {test_metrics['precision']} |
| Recall | {test_metrics['recall']} |
| Brier score | {test_metrics['brier']} |

**Decision threshold:** {threshold} (optimised for F1 on validation set)

## Features

### Input features (19 raw + 2 derived)

| Feature | Type | Description |
|---|---|---|
| tenure | Numerical | Months as a customer |
| MonthlyCharges | Numerical | Current monthly bill ($) |
| TotalCharges | Numerical | Total spend to date ($) |
| charges_per_month | Derived | TotalCharges / (tenure + 1) |
| num_services | Derived | Count of add-on services (0-6) |
| Contract | Categorical | Month-to-month / One year / Two year |
| InternetService | Categorical | DSL / Fiber optic / No |
| PaymentMethod | Categorical | 4 payment types |
| ... | ... | ... |

### Top 5 predictive features (by SHAP importance)
1. tenure — longer tenure strongly reduces churn risk
2. Contract type — two-year contract is the single most protective categorical
3. MonthlyCharges — higher monthly bill increases churn risk
4. charges_per_month — derived feature, outperforms raw TotalCharges
5. num_services — more services = higher switching cost = lower churn

## Limitations

1. **Dataset size:** 7,043 rows is small for a production churn model.
   Performance estimates have meaningful variance — a larger dataset
   would produce more reliable metrics.

2. **Data vintage:** The IBM Telco dataset is synthetic and dated.
   Real telecom data would include call quality metrics, service outages,
   competitor pricing, and customer support interactions — all likely
   more predictive than what's available here.

3. **No temporal validation:** The train/val/test split is random, not
   time-based. In production, models should be validated on future data
   (train on Jan-Sep, validate on Oct-Dec) to catch temporal drift.

4. **Class balance assumption:** The model was trained on 26.5% churn rate.
   If the real churn rate shifts significantly (e.g. during a price
   increase campaign), the calibration will drift and the threshold
   will need to be re-tuned.

5. **Missing features:** No data on customer service calls, network
   quality scores, or competitor offers — all known churn drivers in
   telecom. The model explains what it can from contract and billing data.

## Ethical Considerations

- The model does not use protected characteristics (race, religion,
  national origin) as features.
- Gender is included as a feature but SHAP analysis shows it has
  negligible importance — removing it does not meaningfully change
  performance.
- Recommendations generated by this model should be reviewed by
  humans before action is taken. Automated interventions based solely
  on model output are not recommended.

## Training Infrastructure

- Language: Python 3.13
- Key libraries: LightGBM, scikit-learn, SHAP, MLflow, Optuna
- Training time: < 30 seconds on CPU
- Hyperparameter tuning: Optuna TPE, {config['n_trials']} trials

## How to Use

```python
import mlflow
mlflow.set_tracking_uri('sqlite:///mlflow.db')
model = mlflow.sklearn.load_model('models:/churn-model/Production')
predictions = model.predict_proba(X)[:, 1]
```

## Model Registry

- Registered name: `churn-model`
- Registry: MLflow Model Registry (local SQLite)
- Promotion to Production: manual after validation
"""

with open('reports/model_card.md', 'w') as f:
    f.write(model_card_content)

print("Model card saved to reports/model_card.md")
print(f"Word count: {len(model_card_content.split())}")

C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Model card saved to reports/model_card.md
Word count: 759


C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [11]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('model-comparison')

with mlflow.start_run(run_name='LightGBM_calibrated_final'):

    # Log all test metrics
    for k, v in test_metrics.items():
        mlflow.log_metric(f'test_{k}', v)

    # Log calibration info
    mlflow.log_param('model_name',          'LGBMClassifier')
    mlflow.log_param('calibration',         calib_method)
    mlflow.log_param('optimal_threshold',   threshold)
    mlflow.log_param('tuned_by',            'optuna')

    # Log the model
    mlflow.sklearn.log_model(
        final_pipeline,
        name='pipeline',
        registered_model_name='churn-model',
    )

    # Log artifacts
    mlflow.log_artifact('reports/model_card.md')
    mlflow.log_artifact('models/threshold_config.json')
    mlflow.log_artifact('models/test_metrics.json')
    mlflow.log_artifact('reports/figures/calibration_comparison.png')

    run_id = mlflow.active_run().info.run_id
    print(f"Logged to MLflow. Run ID: {run_id[:8]}")
    print(f"Model registered as 'churn-model'")

2026/06/01 11:43:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'churn-model' already exists. Creating a new version of this model...
Created version '18' of model 'churn-model'.


Logged to MLflow. Run ID: 82c4cb6a
Model registered as 'churn-model'
